## Training of models, that were used in the paper

In [ ]:
import sys, os
sys.path.insert(0, '/home/milan/Downloads/alignn')
sys.path.insert(0, '/home/milan/Downloads/alignn/alignn')

import torch
import numpy as np
import matplotlib.pyplot as plt

from pore_graph_dataset import get_pore_graph_loaders, sanity_check
from alignn.train import train_dgl
from alignn.config import TrainingConfig

config = TrainingConfig(
    dataset     = 'hmof',
    target      = 'surface_area_m2g',
    id_tag      = 'id',
    n_train     = 1103,       # 80% of 1379
    n_val       = 138,        # 10%
    n_test      = 138,        # 10%
    train_ratio = None,
    val_ratio   = None,
    test_ratio  = None,
    batch_size  = 4,
    epochs      = 5,
    keep_data_order = True,   # takes first N structures in order, no shuffling
    output_dir  = './output_option_b',
)
config.model.use_pore_graph = True
config.model.pore_features  = 0

PORE_GRAPHS_PATH = '/home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_1pct.pt'

loaders = get_pore_graph_loaders(config, pore_graphs_path=PORE_GRAPHS_PATH)

sanity_check(loaders[0])

train_dgl(config, train_val_test_loaders=list(loaders))

fatal: not a git repository (or any of the parent directories): .git


Using LMDB dataset.
Obtaining hMOF dataset 137k...
Reference:https://doi.org/10.1021/acs.jpcc.6b08729
Loading the zipfile...
Loading completed.
MAX val: 6947.3
MIN val: 0.0
MAD: 1466.1508943159604
Baseline MAE: 1476.992692524998
data range 6534.1 0.0
line_graph True


100%|██████████| 1103/1103 [00:00<00:00, 2826094.88it/s]


Reading dataset sampletrain_data
data range 6109.8 0.0
line_graph True


100%|██████████| 138/138 [00:00<00:00, 1668051.73it/s]


...

100%|██████████| 138/138 [00:00<00:00, 2607270.05it/s]

Reading dataset sampletest_data
n_train: 1103
n_val  : 138
n_test : 138


Loading precomputed pore graphs from /home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_1pct.pt
  Loaded 1376 entries
Pore-graph splits: train=1103  val=138  test=138
Batch contents:
  g       : DGLGraph   nodes=250  edges=3394
  lg      : DGLGraph   nodes=3394  edges=47350
  lat     : torch.Size([4, 3, 3])
  G_pore  : DGL HeteroGraph
            atom nodes = 250  pore nodes = 37  edges = 347
            atom features: torch.Size([250, 92])
            pore features: torch.Size([37, 2])
  target  : torch.Size([4])

Sanity check passed.
config: {'version': 'NA', 'dataset': 'hmof', 'target': 'surface_area_m2g', 'atom_features': 'cgcnn', 'neighbor_strategy': 'k-nearest', 'id_tag': 'id', 'dtype': 'float32', 'random_seed': 123, 'classification_threshold': None, 'n_val': 138, 'n_test': 138, 'n_train': 1103, 'train_ratio': None, 'val_ratio': None, 'test_ratio': None, 'target_multiplication_factor': None, 'epochs': 5, 'batch_size': 4, 'weight_decay': 0.0, 'learning_rate': 0.01, 'filen

/home/milan/miniconda3/envs/mof/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             0        396836.3209 396836.3209 0.0000     0.0000     0.0000     0.0000     468.73    
Val Loss:    Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             0        41474.0660 41474.0660 0.0000     0.0000     0.0000     0.0000     26.01      Saving model
Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             1        249943.9992 249943.9992 0.0000     0.0000     0.0000     0.0000     425.31    
Val Loss:    Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             1        41839.7112 41839.7112 0.0000     0.0000     0.0000     0.0000     25.38                
Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             2        222197.2604 222197.2604 0.0000     0.0000     0.0000     0.

In [2]:
import json, os
import numpy as np
from sklearn.metrics import mean_absolute_error

results_dir = './output_option_b'

with open(os.path.join(results_dir, 'Val_results.json')) as f:
    test_results = json.load(f)

targets, predictions = [], []
for r in test_results:
    t, p = r['target_out'], r['pred_out']
    if isinstance(t, list):
        targets.extend(t)
        predictions.extend(p)
    else:
        targets.append(t)
        predictions.append(p)

targets     = np.array(targets)
predictions = np.array(predictions)

mae = mean_absolute_error(targets, predictions)
r2  = np.corrcoef(targets, predictions)[0, 1] ** 2

print(f'=== Option B — surface_area_m2g ===')
print(f'  N:   {len(targets)}')
print(f'  MAE: {mae:.2f} m²/g')
print(f'  R²:  {r2:.4f}')

=== Option B — surface_area_m2g ===
  N:   136
  MAE: 1219.83 m²/g
  R²:  0.3569


In [1]:
import sys, os
sys.path.insert(0, '/home/milan/Downloads/alignn')
sys.path.insert(0, '/home/milan/Downloads/alignn/alignn')

import torch
import numpy as np
import matplotlib.pyplot as plt

from pore_graph_dataset import get_pore_graph_loaders, sanity_check
from alignn.train import train_dgl
from alignn.config import TrainingConfig

config = TrainingConfig(
    dataset     = 'hmof',
    target      = 'surface_area_m2g',
    id_tag      = 'id',
    n_train     = 1103,       # 80% of 1379
    n_val       = 138,        # 10%
    n_test      = 138,        # 10%
    batch_size  = 4,
    epochs      = 5,
    learning_rate = 1e-3,       # <-- add this
    keep_data_order = True,   # takes first N structures in order, no shuffling
    output_dir  = './output_option_b2',
)
config.model.use_pore_graph = True
config.model.pore_features  = 0

PORE_GRAPHS_PATH = '/home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_1pct.pt'

loaders = get_pore_graph_loaders(config, pore_graphs_path=PORE_GRAPHS_PATH)

sanity_check(loaders[0])

train_dgl(config, train_val_test_loaders=list(loaders))

fatal: not a git repository (or any of the parent directories): .git


Using LMDB dataset.
Obtaining hMOF dataset 137k...
Reference:https://doi.org/10.1021/acs.jpcc.6b08729
Loading the zipfile...
Loading completed.
MAX val: 6947.3
MIN val: 0.0
MAD: 1466.1508943159604
Baseline MAE: 1476.992692524998
data range 6534.1 0.0
line_graph True


100%|██████████| 1103/1103 [00:00<00:00, 2765282.31it/s]


Reading dataset sampletrain_data
data range 6109.8 0.0
line_graph True


100%|██████████| 138/138 [00:00<00:00, 2296880.76it/s]


...

100%|██████████| 138/138 [00:00<00:00, 2411724.80it/s]

Reading dataset sampletest_data
n_train: 1103
n_val  : 138
n_test : 138


Loading precomputed pore graphs from /home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_1pct.pt
  Loaded 1376 entries
Pore-graph splits: train=1103  val=138  test=138
Batch contents:
  g       : DGLGraph   nodes=487  edges=6894
  lg      : DGLGraph   nodes=6894  edges=100388
  lat     : torch.Size([4, 3, 3])
  G_pore  : DGL HeteroGraph
            atom nodes = 487  pore nodes = 23  edges = 178
            atom features: torch.Size([487, 92])
            pore features: torch.Size([23, 2])
  target  : torch.Size([4])

Sanity check passed.
config: {'version': 'NA', 'dataset': 'hmof', 'target': 'surface_area_m2g', 'atom_features': 'cgcnn', 'neighbor_strategy': 'k-nearest', 'id_tag': 'id', 'dtype': 'float32', 'random_seed': 123, 'classification_threshold': None, 'n_val': 138, 'n_test': 138, 'n_train': 1103, 'train_ratio': 0.8, 'val_ratio': 0.1, 'test_ratio': 0.1, 'target_multiplication_factor': None, 'epochs': 5, 'batch_size': 4, 'weight_decay': 0.0, 'learning_rate': 0.001, 'filena

/home/milan/miniconda3/envs/mof/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             0        687863.3281 687863.3281 0.0000     0.0000     0.0000     0.0000     466.25    
Val Loss:    Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             0        82082.4397 82082.4397 0.0000     0.0000     0.0000     0.0000     24.82      Saving model
Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             1        527684.6163 527684.6163 0.0000     0.0000     0.0000     0.0000     458.06    
Val Loss:    Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             1        57910.0977 57910.0977 0.0000     0.0000     0.0000     0.0000     24.60      Saving model
Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             2        356232.3595 356232.3595 0.0000     0.0000     0.0000     

In [2]:
import json, os
import numpy as np
from sklearn.metrics import mean_absolute_error

results_dir = './output_option_b2'

with open(os.path.join(results_dir, 'Val_results.json')) as f:
    test_results = json.load(f)

targets, predictions = [], []
for r in test_results:
    t, p = r['target_out'], r['pred_out']
    if isinstance(t, list):
        targets.extend(t)
        predictions.extend(p)
    else:
        targets.append(t)
        predictions.append(p)

targets     = np.array(targets)
predictions = np.array(predictions)

mae = mean_absolute_error(targets, predictions)
r2  = np.corrcoef(targets, predictions)[0, 1] ** 2

print(f'=== Option B — surface_area_m2g ===')
print(f'  N:   {len(targets)}')
print(f'  MAE: {mae:.2f} m²/g')
print(f'  R²:  {r2:.4f}')

=== Option B — surface_area_m2g ===
  N:   136
  MAE: 988.27 m²/g
  R²:  0.5139


In [ ]:
import os, sys, torch
sys.path.insert(0, '/home/milan/Downloads/alignn')
sys.path.insert(0, '/home/milan/Downloads/alignn/alignn')
from jarvis.db.figshare import data as jdata
from jarvis.core.atoms import Atoms
from bipartite_pore_graph import build_bipartite_pore_graph
from tqdm import tqdm

OUTPUT_PATH = '/home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_3pct.pt'

if __name__ == '__main__':
    raw    = jdata('hmof')
    N      = 3000
    subset = raw[:N]

    pore_graphs = torch.load(OUTPUT_PATH, weights_only=False) if os.path.exists(OUTPUT_PATH) else {}
    print(f'Resuming: {len(pore_graphs)} done, {N - len(pore_graphs)} remaining')

    for entry in tqdm(subset):
        jid = str(entry['id'])
        if jid in pore_graphs:
            continue
        try:
            G = build_bipartite_pore_graph(Atoms.from_dict(entry['atoms']))
        except Exception:
            G = None
        pore_graphs[jid] = G
        if len(pore_graphs) % 100 == 0:
            torch.save(pore_graphs, OUTPUT_PATH)

    torch.save(pore_graphs, OUTPUT_PATH)
    n_none = sum(1 for v in pore_graphs.values() if v is None)
    print(f'Done. {len(pore_graphs)} total, {n_none} failed/no pores')

In [2]:
import sys, os
sys.path.insert(0, '/home/milan/Downloads/alignn')
sys.path.insert(0, '/home/milan/Downloads/alignn/alignn')

import torch
import numpy as np
import matplotlib.pyplot as plt

from pore_graph_dataset import get_pore_graph_loaders, sanity_check
from alignn.train import train_dgl
from alignn.config import TrainingConfig

config = TrainingConfig(
    dataset     = 'hmof',
    target      = 'surface_area_m2g',
    id_tag      = 'id',
    train_ratio = 0.01,
    val_ratio   = 0.005,
    test_ratio  = 0.005,
    batch_size  = 4,
    epochs      = 5,
    learning_rate = 1e-3,     # add this
    keep_data_order = True,   # takes first N structures in order, no shuffling
    filename      = 'option_b_3pct',
    output_dir  = './output_option_b2',
)
config.model.use_pore_graph = True
config.model.pore_features  = 0

PORE_GRAPHS_PATH = '/home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_3pct.pt'

loaders = get_pore_graph_loaders(config, pore_graphs_path=PORE_GRAPHS_PATH)

sanity_check(loaders[0])

train_dgl(config, train_val_test_loaders=list(loaders))

fatal: not a git repository (or any of the parent directories): .git


Using LMDB dataset.
Obtaining hMOF dataset 137k...
Reference:https://doi.org/10.1021/acs.jpcc.6b08729
Loading the zipfile...
Loading completed.
MAX val: 6947.3
MIN val: 0.0
MAD: 1466.1508943159604
Baseline MAE: 1441.5340029661302
data range 6534.1 0.0
line_graph True


100%|██████████| 1376/1376 [00:51<00:00, 26.93it/s]


data range 6383.5 0.0
line_graph True


100%|██████████| 688/688 [00:25<00:00, 26.73it/s]


data range 6276.5 0.0
line_graph True


100%|██████████| 688/688 [00:24<00:00, 28.02it/s]


n_train: 1376
n_val  : 688
n_test : 688
Loading precomputed pore graphs from /home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_3pct.pt
  Loaded 3000 entries
Pore-graph splits: train=1376  val=688  test=688
Batch contents:
  g       : DGLGraph   nodes=495  edges=6708
  lg      : DGLGraph   nodes=6708  edges=92442
  lat     : torch.Size([4, 3, 3])
  G_pore  : DGL HeteroGraph
            atom nodes = 495  pore nodes = 21  edges = 251
            atom features: torch.Size([495, 92])
            pore features: torch.Size([21, 2])
  target  : torch.Size([4])

Sanity check passed.
config: {'version': 'NA', 'dataset': 'hmof', 'target': 'surface_area_m2g', 'atom_features': 'cgcnn', 'neighbor_strategy': 'k-nearest', 'id_tag': 'id', 'dtype': 'float32', 'random_seed': 123, 'classification_threshold': None, 'n_val': None, 'n_test': None, 'n_train': None, 'train_ratio': 0.01, 'val_ratio': 0.005, 'test_ratio': 0.005, 'target_multiplication_factor': None, 'epochs': 5, 'batch_size': 4, 'weigh

/home/milan/miniconda3/envs/mof/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             0        831544.4856 831544.4856 0.0000     0.0000     0.0000     0.0000     601.53    
Val Loss:    Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             0        350049.3957 350049.3957 0.0000     0.0000     0.0000     0.0000     126.54     Saving model
Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             1        566843.2845 566843.2845 0.0000     0.0000     0.0000     0.0000     591.22    
Val Loss:    Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             1        216830.2241 216830.2241 0.0000     0.0000     0.0000     0.0000     124.62     Saving model
Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             2        361077.7579 361077.7579 0.0000     0.0000     0.0000 

In [4]:
import json, os
import numpy as np
from sklearn.metrics import mean_absolute_error

results_dir = '/home/milan/Downloads/alignn/Thesis/Pore_plan_b/runs/option_b_lr1e3_fixed'

with open(os.path.join(results_dir, 'Val_results.json')) as f:
    test_results = json.load(f)

targets, predictions = [], []
for r in test_results:
    t, p = r['target_out'], r['pred_out']
    if isinstance(t, list):
        targets.extend(t)
        predictions.extend(p)
    else:
        targets.append(t)
        predictions.append(p)

targets     = np.array(targets)
predictions = np.array(predictions)

mae = mean_absolute_error(targets, predictions)
r2  = np.corrcoef(targets, predictions)[0, 1] ** 2

print(f'=== Option B — surface_area_m2g ===')
print(f'  N:   {len(targets)}')
print(f'  MAE: {mae:.2f} m²/g')
print(f'  R²:  {r2:.4f}')

=== Option B — surface_area_m2g ===
  N:   688
  MAE: 862.83 m²/g
  R²:  0.5688


In [1]:
import sys, os
sys.path.insert(0, '/home/milan/Downloads/alignn')
sys.path.insert(0, '/home/milan/Downloads/alignn/alignn')


# Cell 2 — train
from pore_graph_dataset import get_pore_graph_loaders, sanity_check
from alignn.train import train_dgl
from alignn.config import TrainingConfig

config = TrainingConfig(
    dataset         = 'hmof',
    target          = 'surface_area_m2g',
    id_tag          = 'id',
    train_ratio     = 0.10,
    val_ratio       = 0.05,
    test_ratio      = 0.05,
    batch_size      = 16,
    epochs          = 30,
    learning_rate   = 1e-3,
    keep_data_order = True,
    filename        = 'option_b_20pct',
    output_dir      = '/home/milan/Downloads/alignn/Thesis/Pore_plan_b/runs/option_b_20pct',
)
config.model.use_pore_graph = True
config.model.pore_features  = 0

PORE_GRAPHS_PATH = '/home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_20pct.pt'

loaders = get_pore_graph_loaders(config, pore_graphs_path=PORE_GRAPHS_PATH)
sanity_check(loaders[0])
train_dgl(config, train_val_test_loaders=list(loaders))

fatal: not a git repository (or any of the parent directories): .git


Using LMDB dataset.
Obtaining hMOF dataset 137k...
Reference:https://doi.org/10.1021/acs.jpcc.6b08729
Loading the zipfile...
Loading completed.
MAX val: 6947.3
MIN val: 0.0
MAD: 1466.1508943159604
Baseline MAE: 1482.3602690658033
data range 6927.3 0.0
line_graph True


100%|██████████| 13765/13765 [00:00<00:00, 4345194.14it/s]


Reading dataset option_b_20pcttrain_data
data range 6650.5 0.0
line_graph True


100%|██████████| 6882/6882 [00:00<00:00, 2605632.80it/s]


...

Reading dataset option_b_20pcttest_data
n_train: 13765
n_val  : 6882
n_test : 6882


Loading precomputed pore graphs from /home/milan/Downloads/alignn/Thesis/Pore_plan_b/pore_graphs_20pct.pt
  Loaded 27530 entries
Pore-graph splits: train=13765  val=6882  test=6882
Batch contents:
  g       : DGLGraph   nodes=1426  edges=19480
  lg      : DGLGraph   nodes=19480  edges=271632
  lat     : torch.Size([16, 3, 3])
  G_pore  : DGL HeteroGraph
            atom nodes = 1381  pore nodes = 78  edges = 713
            atom features: torch.Size([1381, 92])
            pore features: torch.Size([78, 2])
  target  : torch.Size([16])

Sanity check passed.
config: {'version': 'NA', 'dataset': 'hmof', 'target': 'surface_area_m2g', 'atom_features': 'cgcnn', 'neighbor_strategy': 'k-nearest', 'id_tag': 'id', 'dtype': 'float32', 'random_seed': 123, 'classification_threshold': None, 'n_val': None, 'n_test': None, 'n_train': None, 'train_ratio': 0.1, 'val_ratio': 0.05, 'test_ratio': 0.05, 'target_multiplication_factor': None, 'epochs': 30, 'batch_size': 16, 'weight_decay': 0.0, 'learning_rat

/home/milan/miniconda3/envs/mof/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             0        1426165.9098 1426165.9098 0.0000     0.0000     0.0000     0.0000     5500.53   
Val Loss:    Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             0        301043.0164 301043.0164 0.0000     0.0000     0.0000     0.0000     1215.53    Saving model
Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             1        452359.8885 452359.8885 0.0000     0.0000     0.0000     0.0000     5500.22   
Val Loss:    Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             1        220971.9190 220971.9190 0.0000     0.0000     0.0000     0.0000     1219.31    Saving model
Train Loss:  Epoch    Total      Graph      Atom       Grad       Stress     Addn.      Time      
             2        388332.1977 388332.1977 0.0000     0.0000     0.000

KeyboardInterrupt: 

raw calc: VAL MAE of around 500